In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.alpha_correlations import void_fraction_correlations
from src.combined_properties import properties_df
from src.void_fraction_per_pattern import void_fraction_calculation_function

In [3]:
pd.set_option('display.max_rows', 100)

CORR_COLS = [
    'alpha_h', 'lockhart', 'zivi', 'premoli', 'kanizawa',
    'tibirica', 'hughmark', 'rouhani', 'steiner',
]
T2_MISSING_MAX = 50

# Load data

In [4]:
silver_path = project_root / 'data' / 'silver_layer'
input_path = silver_path / 'confocal_runs'
output_path = project_root / 'data' / 'gold_layer' / 'confocal_void_fraction'
output_path.mkdir(parents=True, exist_ok=True)

df_confocal = pd.read_pickle(silver_path / 'confocal_results.pkl')
df_experimental = pd.read_pickle(silver_path / 'experimental_results.pkl')

# Confocal void fraction (per run)

In [12]:
vf_results = []

for file in sorted(input_path.glob('confocal_run_*.csv')):
    match = re.search(r'confocal_run_(\d+)', file.stem)
    if not match:
        continue

    run_id = int(match.group(1))
    df_run = pd.read_csv(file)

    reduced_pattern = (
        df_experimental.loc[
            df_experimental['run_id'] == run_id,
            'reduced_pattern',
        ].iloc[0]
    )

    void_fraction = void_fraction_calculation_function(df_run, reduced_pattern)

    vf_results.append({
        'run_id': run_id,
        'void_fraction': void_fraction,
    })

df_vf = pd.DataFrame(vf_results).sort_values('run_id').reset_index(drop=True)

c:\Users\fernando\Downloads\projects\phd_paper\src\void_fraction_per_pattern.py:88: RuntimeWarning: Mean of empty slice
  return np.nanmean(vf)
c:\Users\fernando\Downloads\projects\phd_paper\src\void_fraction_per_pattern.py:88: RuntimeWarning: Mean of empty slice
  return np.nanmean(vf)


# Thermophysical properties and void-fraction correlations

In [7]:
df_consolidated = pd.merge(df_experimental, df_vf, on='run_id', how='inner')

df_properties = properties_df(df_consolidated)
df_correlations = void_fraction_correlations(df_properties)

df = pd.merge(
    df_consolidated,
    df_correlations[['run_id'] + CORR_COLS],
    on='run_id',
    how='inner',
)

valid_runs = df_confocal.loc[
    df_confocal['t2_missing_percentage'] < T2_MISSING_MAX,
    'run_id',
]
df = df[df['run_id'].isin(valid_runs)].reset_index(drop=True)

# Comparison: experimental vs correlations

In [ ]:
id_cols = ['run_id', 'reduced_pattern', 'void_fraction']

df_wide = df[id_cols + CORR_COLS].copy()
for col in CORR_COLS:
    df_wide[f'error_{col}'] = df_wide[col] - df_wide['void_fraction']

error_cols = [f'error_{col}' for col in CORR_COLS]
df_wide = df_wide[id_cols + CORR_COLS + error_cols]

df_comparison = df_wide.melt(
    id_vars=id_cols,
    value_vars=CORR_COLS,
    var_name='correlation',
    value_name='void_fraction_corr',
)
_errors = df_wide.melt(
    id_vars=id_cols,
    value_vars=error_cols,
    var_name='correlation',
    value_name='error',
)
_errors['correlation'] = _errors['correlation'].str.removeprefix('error_')

df_comparison = df_comparison.merge(_errors, on=id_cols + ['correlation'])
df_comparison['abs_error'] = df_comparison['error'].abs()

In [17]:
summary = (
    df_comparison
    .groupby(['reduced_pattern', 'correlation'], observed=True)
    .agg(
        n_runs=('run_id', 'count'),
        mae=('abs_error', 'mean'),
        rmse=('error', lambda s: np.sqrt((s ** 2).mean())),
        bias=('error', 'mean'),
    )
    .reset_index()
    .sort_values(['reduced_pattern', 'mae'])
)

# Save

In [ ]:
save_path = project_root / 'data' / 'gold_layer'

df_comparison.to_csv(
    save_path / 'void_fraction_results.csv',
)

df_comparison.to_pickle(
    save_path / 'void_fraction_results.pkl',
)

summary.to_csv(
    save_path / 'void_fraction_grouped_results.csv'
)

summary.to_pickle(
    save_path / 'void_fraction_grouped_results.pkl'
)